# Kiwi · prueba multiobjetos
Grounding DINO tiny → recortes → Qwen3-VL-4B. Experimento independiente; no modifica la API ni los LEDs. Seleccionar GPU T4 y ejecutar en orden. Qwen descarga aproximadamente 9 GB. Sin verificación por EmbeddingGemma en esta prueba.


In [ ]:
%pip install -q -U transformers accelerate pillow gradio


## 1. Cargar Qwen y las categorías actuales


In [ ]:
import time, threading
from PIL import Image, ImageOps
import torch
CATEGORIES = {'soldering_station': 'An electronics soldering and hot air rework station with a control unit and iron holder', 'digital_multimeter': 'A handheld digital multimeter with a screen and rotary selector', 'bench_power_supply': 'A laboratory bench DC power supply with output terminals and controls', 'servo_motor': 'A small blue micro servo motor with a three-wire cable', 'oscilloscope': 'A digital oscilloscope with a waveform screen and input connectors', 'function_generator': 'A laboratory function generator with frequency controls and signal outputs', 'variable_capacitor': 'A laboratory variable parallel plate capacitor with circular metal plates on a base', 'capacitors': 'A small cylindrical electrolytic capacitor with two wire leads', 'power_adapter': 'A rectangular AC DC power adapter with an integrated output cable', 'multimeter_box': 'A yellow orange cardboard retail box for an AstroAI digital multimeter', 'solder': 'An electric soldering iron with a handle and pointed metal tip', 'otros': 'An everyday object that is not one of the laboratory tools, such as household items, hand tools or bare circuit boards'}
LOCATION_BY_CATEGORY = {}
reference_vectors = None
def prepare_image(image):
    image = ImageOps.exif_transpose(image).convert('RGB')
    image.thumbnail((1024, 1024))
    return image
def identify(image):
    raise RuntimeError('Reconocedor de referencias no cargado en esta prueba.')
# Kiwi · reconocimiento con un modelo de visión y lenguaje (VLM), sin fotos de referencia.
#
# Se ejecuta en el notebook después de las secciones 1–4 (usa CATEGORIES, decide e identify):
#     exec(open('/content/kiwi_vlm.py').read())
# Reemplaza `identify`, así que la API REST de la sección 6b y la página de Gradio pasan a usar el VLM
# sin volver a ejecutarlas. Para volver a EmbeddingGemma: identify = identify_embeddings
#
# El VLM recibe la foto y una pregunta de opción múltiple con las categorías; se leen las probabilidades
# de cada letra de respuesta. `similarity` pasa a ser esa probabilidad (0–1) y `margin` la diferencia con
# la segunda opción. La opción "ninguna" se devuelve como la categoría `otros`.
import time
import torch
from transformers import AutoModelForImageTextToText, AutoProcessor

VLM_ID = 'Qwen/Qwen3-VL-4B-Instruct'
VLM_MIN_PROBABILITY = 0.90   # recognized=True solo por encima de esta probabilidad...
VLM_MIN_MARGIN = 0.50        # ...y con esta ventaja sobre la segunda opción
# Doble verificación: si la categoría tiene fotos de referencia (sección 5) y EmbeddingGemma elige otra
# herramienta, no se acepta. Evita LEDs equivocados con fotos ambiguas (p. ej. equipos vistos por detrás).
VLM_CHECK_WITH_REFERENCES = False
OTHER = 'otros'

if 'identify_embeddings' not in globals():
    identify_embeddings = identify   # el reconocimiento por fotos de referencia sigue disponible

if 'vlm_model' not in globals():
    print('Cargando', VLM_ID, '(la primera vez descarga ~9 GB)...')
    vlm_processor = AutoProcessor.from_pretrained(
        VLM_ID, min_pixels=256 * 28 * 28, max_pixels=640 * 28 * 28)  # fotos de ~0,5 MP: rápido y suficiente
    vlm_model = AutoModelForImageTextToText.from_pretrained(
        VLM_ID, dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map='auto', attn_implementation='sdpa')
    vlm_model.eval()
    print('VLM cargado. Memoria GPU usada: %.1f GB' % (torch.cuda.memory_allocated() / 1e9))

# Descripciones para el VLM: rasgos del panel frontal que distinguen equipos parecidos por fuera.
# Las categorías sin entrada aquí usan su texto de CATEGORIES.
VLM_DESCRIPTIONS = {
    'oscilloscope': 'Oscilloscope: bench instrument whose front has a large screen with a grid showing waveforms, '
                    'many knobs and BNC probe inputs labeled CH1/CH2',
    'function_generator': 'Function generator: bench instrument whose front has a numeric frequency display, '
                          'waveform buttons (sine, square, triangle) and BNC signal output connectors',
    'bench_power_supply': 'Bench DC power supply: bench instrument whose front shows voltage (V) and current (A) '
                          'readouts, adjustment knobs and red/black banana output terminals',
    'soldering_station': 'Soldering station: control unit with a temperature display or dial, connected to a '
                         'soldering iron and an iron holder stand',
    'power_adapter': 'AC/DC power adapter: small black plastic brick with an attached cable, used to power a '
                     'device from the wall outlet',
    'digital_multimeter': 'Handheld digital multimeter: portable meter with an LCD screen, a large rotary selector '
                          'dial and probe jacks (COM, V, A)',
    'solder': 'Soldering iron: pen-shaped hand tool with a plastic handle and a long pointed metal tip',
}

VLM_OPTIONS = [name for name in CATEGORIES if name != OTHER] + [OTHER]
VLM_LETTERS = [chr(ord('A') + i) for i in range(len(VLM_OPTIONS))]
VLM_LETTER_IDS = [vlm_processor.tokenizer.encode(letter, add_special_tokens=False)[0] for letter in VLM_LETTERS]
assert len(set(VLM_LETTER_IDS)) == len(VLM_LETTERS), 'Cada letra debe ser un token distinto.'


def vlm_prompt():
    lines = [
        'You are checking which tool someone is returning to an electronics lab shelf.',
        'Look only at the main object in the photo; ignore hands, tables, cables in the background and other clutter.',
        'Many bench instruments are similar boxes: decide by what is visible on their front panel. If only the back, '
        'bottom or side of an instrument is visible and you cannot see its front panel, choose the last option.',
        'Options:',
    ]
    for letter, name in zip(VLM_LETTERS, VLM_OPTIONS):
        if name == OTHER:
            lines.append(f'{letter}. None of the above or cannot tell: the main object is something else (for '
                         'example a resistor, LED, loose wire, Arduino or other circuit board, pliers, screwdriver, '
                         'phone or household item), there is no clear object, or only the back/side of an '
                         'instrument is visible.')
        else:
            lines.append(f'{letter}. {VLM_DESCRIPTIONS.get(name, CATEGORIES[name])}.')
    lines.append('Answer with only the letter of the correct option.')
    return '\n'.join(lines)


@torch.inference_mode()
def vlm_scores(image):
    messages = [{'role': 'user', 'content': [{'type': 'image', 'image': image},
                                             {'type': 'text', 'text': vlm_prompt()}]}]
    inputs = vlm_processor.apply_chat_template(messages, tokenize=True, add_generation_prompt=True,
                                               return_dict=True, return_tensors='pt').to(vlm_model.device)
    logits = vlm_model(**inputs).logits[0, -1].float()
    probabilities = torch.softmax(logits[VLM_LETTER_IDS], dim=0).tolist()
    return dict(zip(VLM_OPTIONS, probabilities))


def identify_vlm(image):
    if image is None:
        return {'recognized': False, 'candidate': None, 'location_id': None, 'reason': 'Sube una fotografía.'}
    start = time.perf_counter()
    scores = vlm_scores(prepare_image(image))
    ranking = sorted(scores.items(), key=lambda item: item[1], reverse=True)
    candidate, probability = ranking[0]
    margin = probability - ranking[1][1]
    result = {'candidate': candidate, 'similarity': probability, 'margin': margin, 'mode': 'vlm',
              'model': VLM_ID, 'recognized': False}
    if candidate == OTHER:
        result['reason'] = 'No es una de las herramientas o no se ve su frente. Muestra el frente del objeto.'
    elif probability < VLM_MIN_PROBABILITY:
        result['reason'] = 'Confianza insuficiente: foto poco clara u objeto desconocido.'
    elif margin < VLM_MIN_MARGIN:
        result['reason'] = 'Duda entre dos herramientas. Repite la foto.'
    else:
        result.update(recognized=True, reason='Reconocido por el VLM.')
        if (VLM_CHECK_WITH_REFERENCES and globals().get('reference_vectors') is not None
                and candidate in set(reference_labels)):
            check = identify_embeddings(image)
            result['reference_candidate'] = check['candidate']
            result['reference_similarity'] = check['similarity']
            if check['candidate'] != candidate:
                result.update(recognized=False, reason='El VLM y las fotos de referencia no coinciden. '
                                                        'Muestra el frente de la herramienta.')
    result['location_id'] = LOCATION_BY_CATEGORY.get(candidate) if result['recognized'] else None
    result['elapsed_seconds'] = round(time.perf_counter() - start, 3)
    result['ranking'] = [{'object': name, 'similarity': round(p, 4)} for name, p in ranking]
    return result


identify = identify_vlm
_warm = identify_vlm(Image.new('RGB', (448, 448), 'white'))   # primera inferencia (compila kernels)
print('Reconocimiento activo: VLM', VLM_ID, '| opciones:', ', '.join(f'{l}={n}' for l, n in zip(VLM_LETTERS, VLM_OPTIONS)))
print('Prueba con imagen en blanco ->', _warm['candidate'], round(_warm['similarity'], 3), f"({_warm['elapsed_seconds']} s)")


## 2. Localización y recortes
Basado en https://huggingface.co/docs/transformers/model_doc/grounding-dino. La puntuación de DINO no equivale a la de Qwen. Eliminamos cuadros muy similares, pero los objetos contenidos en otros pueden aparecer por separado. Máximo 12 recortes por foto.


In [ ]:
"""Prueba experimental: Grounding DINO localiza y Qwen clasifica recortes.

No sustituye identify ni modifica la API o las luces de la estación.
"""
import inspect
import math
import time
from PIL import ImageDraw, ImageOps

DETECTOR_ID = 'IDEA-Research/grounding-dino-tiny'
DETECTION_PROMPT = (
    'a soldering station. a digital multimeter. a laboratory power supply. '
    'a servo motor. an oscilloscope. a function generator. a variable capacitor. '
    'a capacitor. a power adapter. a yellow multimeter box. a soldering iron.'
)


def intersection_over_union(a, b):
    intersection = max(0, min(a[2], b[2]) - max(a[0], b[0])) * max(0, min(a[3], b[3]) - max(a[1], b[1]))
    areas = [(x[2] - x[0]) * (x[3] - x[1]) for x in (a, b)]
    union = sum(areas) - intersection
    return intersection / union if union > 0 else 0


def select_detections(detections, size, limit=12, iou_threshold=.6):
    """Clamp boxes and remove near duplicates across detector labels."""
    width, height = size
    selected = []
    for item in sorted(detections, key=lambda x: x['score'], reverse=True):
        if not all(math.isfinite(v) for v in [item['score'], *item['box']]):
            continue
        x1, y1, x2, y2 = item['box']
        box = [max(0, min(width, math.floor(x1))), max(0, min(height, math.floor(y1))),
               max(0, min(width, math.ceil(x2))), max(0, min(height, math.ceil(y2)))]
        if box[2] <= box[0] or box[3] <= box[1]:
            continue
        if any(intersection_over_union(box, old['box']) > iou_threshold for old in selected):
            continue
        selected.append(dict(item, box=box))
        if len(selected) >= limit:
            break
    return selected


class MultiObjectProbe:
    def __init__(self, classify):
        import torch
        from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection
        self.classify = classify
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        self.processor = AutoProcessor.from_pretrained(DETECTOR_ID)
        self.detector = AutoModelForZeroShotObjectDetection.from_pretrained(DETECTOR_ID).to(self.device).eval()

    def detect(self, image, threshold=.25, text_threshold=.2, limit=12):
        import torch
        inputs = self.processor(images=image, text=DETECTION_PROMPT, return_tensors='pt').to(self.device)
        with torch.inference_mode():
            outputs = self.detector(**inputs)
        postprocess = self.processor.post_process_grounded_object_detection
        # Transformers versions use either threshold or box_threshold.
        threshold_name = 'threshold' if 'threshold' in inspect.signature(postprocess).parameters else 'box_threshold'
        result = postprocess(outputs, inputs.input_ids, **{threshold_name: float(threshold)},
                             text_threshold=float(text_threshold), target_sizes=[image.size[::-1]])[0]
        labels = result.get('text_labels', result.get('labels'))
        raw = [{'box': box.tolist(), 'score': float(score), 'label': str(label)}
               for box, score, label in zip(result['boxes'], result['scores'], labels)]
        return select_detections(raw, image.size, limit), len(raw)

    def identify(self, image, threshold=.25):
        if image is None:
            return None, [], {'objects': [], 'reason': 'Sube una foto.'}
        image = ImageOps.exif_transpose(image).convert('RGB')
        start = time.perf_counter()
        detections, raw_count = self.detect(image, threshold)
        detection_seconds = time.perf_counter() - start
        annotated = image.copy()
        draw = ImageDraw.Draw(annotated)
        objects, crops = [], []
        for index, detection in enumerate(detections, 1):
            box = detection['box']
            # A small border preserves context without including most neighbors.
            pad = max(2, round(min(box[2] - box[0], box[3] - box[1]) * .04))
            crop_box = [max(0, box[0] - pad), max(0, box[1] - pad),
                        min(image.width, box[2] + pad), min(image.height, box[3] + pad)]
            crop = image.crop(crop_box)
            result = self.classify(crop)
            accepted = bool(result.get('recognized'))
            candidate = result.get('candidate')
            caption = f'{index}: {candidate or "desconocido"}' + ('' if accepted else ' (revisar)')
            color = 'green' if accepted else 'orange'
            draw.rectangle(box, outline=color, width=max(2, image.width // 400))
            draw.text((box[0], max(0, box[1] - 14)), caption, fill=color)
            crops.append((crop, caption))
            objects.append({'id': index, 'box': box, 'crop_box': crop_box,
                            'detector_label': detection['label'], 'detector_score': detection['score'],
                            'category': candidate, 'recognized': accepted, 'recognition': result})
        return annotated, crops, {'objects': objects, 'raw_detections': raw_count,
                                  'detection_seconds': round(detection_seconds, 3),
                                  'elapsed_seconds': round(time.perf_counter() - start, 3),
                                  'limit': 12, 'reason': 'Sin objetos detectados.' if not objects else
                                  'Prueba experimental: revisar omisiones, duplicados y componentes del mismo equipo.'}


## 3. Prueba reproducible
Tres fotografías del dataset y un montaje artificial. No mide precisión en escenas reales ni es un conjunto de evaluación independiente. Guarda resultados en `/content/kiwi_multiobject_results.json`.


In [ ]:
import io, json, urllib.request
from pathlib import Path
samples = {}
for category in ['digital_multimeter', 'servo_motor', 'solder']:
    url = f'https://raw.githubusercontent.com/fcanas574/kiwi/main/dataset_prepared/references/{category}/{category}_001.jpg'
    samples[category] = ImageOps.exif_transpose(Image.open(io.BytesIO(urllib.request.urlopen(url).read()))).convert('RGB')
# Montage of three separate photographs: diagnostic only, not a real multiobject scene.
montage = Image.new('RGB', (1200, 400), 'white')
for i, photo in enumerate(samples.values()):
    photo = photo.copy(); photo.thumbnail((390, 390))
    montage.paste(photo, (i * 400 + (400-photo.width)//2, (400-photo.height)//2))
probe = MultiObjectProbe(identify_vlm)
outputs = []
for name, photo in [*samples.items(), ('montage_artificial', montage)]:
    annotated, crops, result = probe.identify(photo)
    print(name, json.dumps(result, ensure_ascii=False))
    display(annotated)
    outputs.append({'sample': name, **result})
Path('/content/kiwi_multiobject_results.json').write_text(json.dumps(outputs, indent=2, ensure_ascii=False))


## 4. Probar una foto real
Subir una imagen con varias herramientas. Revisar objetos omitidos, duplicados, confusiones y tiempo. Para el celular se puede usar la subida de archivos de Colab; no se publica un nuevo enlace.


In [ ]:
import gradio as gr
model_lock = threading.Lock()
def identify_multiple(image, threshold):
    with model_lock:
        return probe.identify(image, threshold)
multi_demo = gr.Interface(fn=identify_multiple,
    inputs=[gr.Image(type='pil', label='Foto con varias herramientas'),
            gr.Slider(.1, .6, value=.25, step=.01, label='Umbral de detección')],
    outputs=[gr.Image(label='Objetos localizados'), gr.Gallery(label='Recortes y categorías'), gr.JSON(label='Resultados')],
    title='Kiwi · prueba Grounding DINO + Qwen',
    description='Verde: aceptado por Qwen. Naranja: revisar. No activa LEDs. El montaje de prueba no sustituye una foto real.')
multi_demo.queue(default_concurrency_limit=1).launch(share=False)
